In [4]:
import os
import mne
import pyedflib
import numpy as np
import scipy
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import jupyterlab
import yasa
import glob
import gc

# Configuration for interactive plots
%matplotlib widget

# Safe import of psutil with error handling in case it is not installed
try:    
    import psutil
except ImportError:    
    psutil = None

# Definition of the RAM check function
def print_ram(label=""):
    if psutil is None:
        print(f"{label} RAM: unavailable (psutil not installed)")
        return
        
    process = psutil.Process(os.getpid())
    mb = process.memory_info().rss / 1024 / 1024
    print(f"{label} RAM: {mb:.1f} MB")


print_ram("Initial state")

Initial state RAM: 454.7 MB


In [2]:

# Path to the folder containing your .edf files (change to your actual path)
data_folder = "/mnt/c/Users/olivi/Desktop/___EEG/CZD_sampleData/CZD_sampleData/eeg_edf"

# Retrieve a list of paths to all .edf files in that folder
edf_files = glob.glob(f"{data_folder}/*.edf")

# Display the found files and their total count
print(f"Found {len(edf_files)} EDF files.")
for file in edf_files:
    print(file)

Found 3 EDF files.
/mnt/c/Users/olivi/Desktop/___EEG/CZD_sampleData/CZD_sampleData/eeg_edf/2023-07-26{6591E658-D0E0-49FE-BDE4-FC101EACF45D} Data.edf
/mnt/c/Users/olivi/Desktop/___EEG/CZD_sampleData/CZD_sampleData/eeg_edf/2023-07-27{891F8634-3CC8-4FEE-B1D3-64394DC86F7A} Data.edf
/mnt/c/Users/olivi/Desktop/___EEG/CZD_sampleData/CZD_sampleData/eeg_edf/2023-07-27{D29553B9-1220-4DFC-8456-F712EE2281C8} Data.edf


In [9]:
def create_bipolar(raw_input):

    electrodes = {}

    for ch in raw_input.ch_names:
        if not ch[-1].isdigit():
            continue

        electrode = ch.rstrip("0123456789")

        if electrode not in electrodes:
            electrodes[electrode] = []

        electrodes[electrode].append(ch)

    anode = []
    cathode = []
    bipolar_names = []

    for electrode in electrodes:
        # Sort channels by their actual number (integer value)
        channels = sorted(
            electrodes[electrode],
            key=lambda x: int(x[len(electrode):])
        )

        for i in range(len(channels) - 1):
            # Extract numerical numbers for the current and next channel
            current_num = int(channels[i][len(electrode):])
            next_num = int(channels[i + 1][len(electrode):])
            
            # CONDITION: Create a pair only if they are adjacent (the difference is exactly 1)
            if next_num - current_num == 1:
                anode.append(channels[i])
                cathode.append(channels[i + 1])
                bipolar_names.append(f"{channels[i]}-{channels[i + 1]}")
            else:
                # Optional: you can add a print statement here, e.g., print(f"Skipped gap: {channels[i]} to {channels[i+1]}")
                continue

    raw_bipolar = mne.set_bipolar_reference(
        raw_input,
        anode=anode,
        cathode=cathode,
        ch_name=bipolar_names,
        drop_refs=False,
        copy=True,
        verbose=False
    )
 
    return raw_bipolar, bipolar_names

In [11]:


# Pętla po plikach EDF
for edf_file in edf_files:
    file_base_name = os.path.splitext(os.path.basename(edf_file))[0]
    
    # Krok 1: Bardzo wyraźny print o rozpoczęciu pliku
    print("\n" + "="*60)
    print(f'""""""""PROCESSING THIS FILE: {file_base_name} .......................""""""""')
    print("="*60)
    
    output_dir = f"SOZ/{file_base_name}"
    os.makedirs(output_dir, exist_ok=True)
    
    # Wczytanie metadanych po cichu
    raw = mne.io.read_raw_edf(edf_file, preload=False, verbose=False)

    # Ekstrakcja adnotacji
    annotations = pd.DataFrame({
        "onset_s": raw.annotations.onset,
        "duration_s": raw.annotations.duration,
        "description": raw.annotations.description
    })

    # Zapis tabeli i informacja o tym w konsoli
    annotations_path = f"{output_dir}/inventory_table_annotations.csv"
    annotations.to_csv(annotations_path, index=False)
    print(f"[INFO] Saved full inventory table to: {annotations_path}")

    # Szukanie napadów
    seizure_rows = annotations[annotations["description"].str.contains("NAPAD", na=False)]

    if seizure_rows.empty:
        print(f"[RESULT] No seizure ('NAPAD') found in this file. Skipping.")
        continue  

    # Informacja o liczbie znalezionych napadów
    total_seizures = len(seizure_rows)
    print(f"[RESULT] Found {total_seizures} seizure event(s) ('NAPAD') in this file.")

    # Pętla po napadach
    for index, row in enumerate(seizure_rows.itertuples(), start=1):
        onset = row.onset_s
        tmin = onset - 10
        tmax = onset + 30

        seizure_output_dir = f"{output_dir}/seizure_{index}"
        os.makedirs(seizure_output_dir, exist_ok=True)
        os.makedirs(f"{seizure_output_dir}/spectrograms", exist_ok=True)

        # Wyciszone wycinanie danych
        raw_cropped = raw.copy().crop(tmin=tmin, tmax=tmax, verbose=False)
        raw_cropped.load_data(verbose=False)

        # Wyciszone filtrowanie
        raw_filtered = raw_cropped.copy()
        raw_filtered.notch_filter(freqs=[50, 100, 150], fir_design='firwin', verbose=False)
        raw_filtered.filter(l_freq=1.0, h_freq=70.0, fir_design='firwin', verbose=False)

        # Montaż bipolarny
        raw_bipolar_filtered, bipolar_names = create_bipolar(raw_filtered)
        total_channels = len(bipolar_names)

        # Informacja o przefiltrowanych kanałach bipolarnych dla tego napadu
        print(f"  -> Seizure #{index}/{total_seizures} | Onset: {onset:.1f}s | Window: {tmin:.1f}s to {tmax:.1f}s | Channels filtered: {total_channels}")

        # Wyciszone rysowanie fali
        fig_filtered = raw_bipolar_filtered.plot(
            picks=bipolar_names, n_channels=15, duration=10.0, scalings="auto",
            title=f"FILTERED Bipolar trace - Seizure {index}", show=False, verbose=False
        )
        fig_filtered.savefig(f"{seizure_output_dir}/bipolar_filtered_trace.png", dpi=300)
        plt.close(fig_filtered)

        # Wyciszone obliczanie PSD
        psd = raw_bipolar_filtered.compute_psd(
            fmin=0,
            fmax=100,
            picks=bipolar_names,
            verbose='ERROR'
        )

        fig_psd = psd.plot(
            average=False,
            show=False
        )
        fig_psd.suptitle(
            f"PSD - bipolar channels - Seizure {index}",
            fontsize=14
        )
        fig_psd.savefig(
            f"{seizure_output_dir}/psd_bipolar.png",
            dpi=300,
            bbox_inches="tight"    
        )   
        plt.close(fig_psd)
        
        # Potwierdzenie zapisu okna i PSD
        print(f"     [SUCCESS] Saved 40s window trace and PSD graph.")
        """
        # Generowanie spektrogramów (czysty scipy/matplotlib, bez logów)
        sfreq = raw_bipolar_filtered.info["sfreq"]
        for channel_for_spectrogram in bipolar_names:
            signal = raw_bipolar_filtered.get_data(picks=[channel_for_spectrogram], verbose=False)[0]
            f, t, Sxx = scipy.signal.spectrogram(signal, fs=sfreq, nperseg=512, noverlap=256)

            fig_spec = plt.figure(figsize=(10, 5))
            plt.pcolormesh(t, f, 10 * np.log10(Sxx), shading="gouraud")
            plt.ylim(0, 100)
            plt.xlabel("Time [s]")
            plt.ylabel("Frequency [Hz]")
            plt.title(f"Spectrogram - {channel_for_spectrogram} (Seizure {index})")
            plt.colorbar(label="Power [dB]")
            plt.tight_layout()

            fig_spec.savefig(f"{seizure_output_dir}/spectrograms/spectrogram_{channel_for_spectrogram}.png", dpi=300)
            plt.close(fig_spec)
        """
        # Czyszczenie pamięci
        del raw_cropped, raw_filtered, raw_bipolar_filtered, psd
        plt.close('all')
        gc.collect()

    print(f"\n[DONE] Finished processing file: {file_base_name}")


""""""""PROCESSING THIS FILE: 2023-07-26{6591E658-D0E0-49FE-BDE4-FC101EACF45D} Data .......................""""""""
[INFO] Saved full inventory table to: SOZ/2023-07-26{6591E658-D0E0-49FE-BDE4-FC101EACF45D} Data/inventory_table_annotations.csv
[RESULT] Found 6 seizure event(s) ('NAPAD') in this file.
  -> Seizure #1/6 | Onset: 15886.7s | Window: 15876.7s to 15916.7s | Channels filtered: 103
Plotting power spectral density (dB=True).


/tmp/ipykernel_1065/787165301.py:81: RuntimeWarning: Channel locations not available. Disabling spatial colors.
  fig_psd = psd.plot(


     [SUCCESS] Saved 40s window trace and PSD graph.
  -> Seizure #2/6 | Onset: 19941.6s | Window: 19931.6s to 19971.6s | Channels filtered: 103
Plotting power spectral density (dB=True).


/tmp/ipykernel_1065/787165301.py:81: RuntimeWarning: Channel locations not available. Disabling spatial colors.
  fig_psd = psd.plot(


     [SUCCESS] Saved 40s window trace and PSD graph.
  -> Seizure #3/6 | Onset: 20373.8s | Window: 20363.8s to 20403.8s | Channels filtered: 103
Plotting power spectral density (dB=True).


...

     [SUCCESS] Saved 40s window trace and PSD graph.
  -> Seizure #6/6 | Onset: 41346.1s | Window: 41336.1s to 41376.1s | Channels filtered: 103
Plotting power spectral density (dB=True).


/tmp/ipykernel_1065/787165301.py:81: RuntimeWarning: Channel locations not available. Disabling spatial colors.
  fig_psd = psd.plot(


     [SUCCESS] Saved 40s window trace and PSD graph.

[DONE] Finished processing file: 2023-07-26{6591E658-D0E0-49FE-BDE4-FC101EACF45D} Data

""""""""PROCESSING THIS FILE: 2023-07-27{891F8634-3CC8-4FEE-B1D3-64394DC86F7A} Data .......................""""""""
[INFO] Saved full inventory table to: SOZ/2023-07-27{891F8634-3CC8-4FEE-B1D3-64394DC86F7A} Data/inventory_table_annotations.csv
[RESULT] Found 1 seizure event(s) ('NAPAD') in this file.
  -> Seizure #1/1 | Onset: 2525.0s | Window: 2515.0s to 2555.0s | Channels filtered: 103
Plotting power spectral density (dB=True).


/tmp/ipykernel_1065/787165301.py:81: RuntimeWarning: Channel locations not available. Disabling spatial colors.
  fig_psd = psd.plot(


     [SUCCESS] Saved 40s window trace and PSD graph.

[DONE] Finished processing file: 2023-07-27{891F8634-3CC8-4FEE-B1D3-64394DC86F7A} Data

""""""""PROCESSING THIS FILE: 2023-07-27{D29553B9-1220-4DFC-8456-F712EE2281C8} Data .......................""""""""
[INFO] Saved full inventory table to: SOZ/2023-07-27{D29553B9-1220-4DFC-8456-F712EE2281C8} Data/inventory_table_annotations.csv
[RESULT] No seizure ('NAPAD') found in this file. Skipping.
